<div style="font-size:11px; line-height:1.4">
<b style="font-size:13px">YOLO11-nano 訓練 (Google Colab 免費 GPU)</b><br>
本機 Intel UHD Graphics 630 無法有效加速，改用 Colab 免費 GPU (Tesla T4)，最後下載 <b>model.pt</b> 回本機。<br>
含雜訊/模糊增強，並<b>特別強化 motorcycle 召回率</b>（過採樣 4x + imgsz 1280 + 高 cls 損失 + 強增強）。<br><br>
<b style="font-size:12px">使用步驟</b><br>
1. 本機先打包：PowerShell 執行 <code>Compress-Archive -Path dataset -DestinationPath dataset.zip -Force</code><br>
2. 開啟 <a href="https://colab.research.google.com/">Google Colab</a>，上傳本檔 <code>train_colab.ipynb</code>。<br>
3. 選單 Runtime → Change runtime type → Hardware accelerator → <b>T4 GPU</b> → Save。<br>
4. 由上而下執行；在「上傳」cell 選擇 <code>dataset.zip</code>。<br>
5. 完成後瀏覽器會自動下載 <code>model.pt</code>，放回本機專案根目錄。<br>
<i>註：imgsz 1280 在 T4 上較慢，屬正常。</i>
</div>

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">1. 確認 GPU</b><br>應顯示 Tesla T4。若否，回 Runtime → Change runtime type 設定 T4 GPU。</div>

In [ ]:
!nvidia-smi

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">2. 安裝 Ultralytics 與 Albumentations</b></div>

In [ ]:
!pip install -q ultralytics albumentations

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">3. 上傳並解壓 dataset.zip</b><br>執行後點 Choose Files 選擇本機 <code>dataset.zip</code>，解壓到 <code>/content/dataset</code>。<br>替代：資料較大可改用 Drive — <code>from google.colab import drive; drive.mount('/content/drive')</code> 再把 DATASET 指向 Drive 路徑。</div>

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

uploaded = files.upload()                  # 選擇 dataset.zip
zip_name = next(iter(uploaded))            # 取得上傳的檔名

with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')               # → /content/dataset

DATASET = Path('/content/dataset')
assert (DATASET / 'data.yaml').exists(), f'解壓後找不到 {DATASET}/data.yaml，請確認 zip 內含 dataset 資料夾'
print('資料集就緒:', DATASET)

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">4. 產生修正版 data.yaml</b><br>原始 <code>data.yaml</code> 用相對路徑且無 <code>path:</code> 鍵，易解析錯誤。用絕對路徑 (<code>/content/dataset</code>) 重寫一份 <code>data_fixed.yaml</code>，不改原檔。</div>

In [ ]:
import yaml

cfg = yaml.safe_load((DATASET / 'data.yaml').read_text(encoding='utf-8'))
fixed = {
    'path': str(DATASET),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': cfg['nc'],
    'names': cfg['names'],
}
FIXED_YAML = DATASET / 'data_fixed.yaml'
FIXED_YAML.write_text(yaml.safe_dump(fixed, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(FIXED_YAML.read_text(encoding='utf-8'))

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">5. 強化 motorcycle：影像過採樣 4x</b><br>掃描 train 標註，含 motorcycle (class 2) 的影像在每個 epoch 重複取樣 4 次，提升其相對頻率以改善召回。產生 <code>train_oversample.txt</code> 並把 yaml 的 <code>train</code> 指向它（Ultralytics 支援以影像清單 .txt 當 train，重複行＝重複取樣）。調整 <code>OVERSAMPLE</code> 可改變倍率。</div>

In [ ]:
import random

MOTO = cfg['names'].index('motorcycle')   # = 2
OVERSAMPLE = 4                            # 含 motorcycle 的影像重複次數
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

img_dir = DATASET / 'train' / 'images'
lbl_dir = DATASET / 'train' / 'labels'

lines, n_moto = [], 0
for img in sorted(img_dir.iterdir()):
    if img.suffix.lower() not in IMG_EXT:
        continue
    lbl = lbl_dir / (img.stem + '.txt')
    has_moto = lbl.exists() and any(
        ln.split()[0] == str(MOTO) for ln in lbl.read_text().splitlines() if ln.strip()
    )
    reps = OVERSAMPLE if has_moto else 1
    lines += [str(img.resolve())] * reps
    n_moto += has_moto

random.seed(0)
random.shuffle(lines)
TRAIN_LIST = DATASET / 'train_oversample.txt'
TRAIN_LIST.write_text('\n'.join(lines), encoding='utf-8')

fixed['train'] = str(TRAIN_LIST.resolve())               # 改用過採樣清單
FIXED_YAML.write_text(yaml.safe_dump(fixed, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(f'含 motorcycle 影像: {n_moto}，過採樣後訓練樣本數: {len(lines)} (原 164)')

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">6. 資料增強設定 (雜訊 / 模糊)</b><br>包裝 Ultralytics 的 Albumentations 注入 Blur、MedianBlur、GaussNoise、ISONoise。屬非空間變換，只改影像、不動 bbox。調整 <code>p</code> 改變機率；其餘內建增強仍保留。須在訓練前執行。</div>

In [ ]:
import albumentations as A
from ultralytics.data import augment as _aug

# 保存原始 __init__，避免重複執行本 cell 時把包裝後的版本當成原始版
_orig_init = getattr(_aug.Albumentations, '_orig_init', _aug.Albumentations.__init__)
_aug.Albumentations._orig_init = _orig_init

def _patched_init(self, p=1.0, transforms=None):   # 接受 Ultralytics 傳入的 transforms 並改用自訂清單
    custom = [
        A.Blur(blur_limit=7, p=0.10),         # 一般模糊
        A.MedianBlur(blur_limit=7, p=0.10),   # 中值模糊
        A.GaussNoise(p=0.20),                 # 高斯雜訊
        A.ISONoise(p=0.10),                   # 感光雜訊
    ]
    _aug.Albumentations._orig_init(self, p=p, transforms=custom)

_aug.Albumentations.__init__ = _patched_init
print('已注入雜訊/模糊增強，訓練 log 的 albumentations: 行會列出 Blur/MedianBlur/GaussNoise/ISONoise')

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">7. 載入 nano 模型並用 GPU 訓練 (強化設定)</b><br><code>device=0</code> T4 GPU；首次自動下載 <code>yolo11n.pt</code>。<br>強化 motorcycle 召回：<code>imgsz=1280</code>(小物件)、<code>cls=1.0</code>(分類損失)、<code>scale/mosaic/mixup</code>(強增強)、<code>patience=30</code>(early stopping)。</div>

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')           # nano 尺寸
results = model.train(
    data=str(FIXED_YAML),
    epochs=150,
    patience=30,                     # early stopping
    imgsz=1280,                      # 高解析度，利於小物件 (motorcycle) 召回
    cls=1.0,                         # 提高分類損失權重 (預設 0.5)
    scale=0.9, mosaic=1.0, mixup=0.1,  # 加強尺度/拼接/混合增強
    device=0,
    project='/content/runs',
    name='train',
    exist_ok=True,
)

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">8. 輸出 model.pt 並下載回本機</b></div>

In [ ]:
import shutil

best = Path(results.save_dir) / 'weights' / 'best.pt'
target = Path('/content/model.pt')
assert best.exists(), f'找不到 best.pt: {best}'
shutil.copy(best, target)
print('已輸出:', target)

files.download(str(target))          # 觸發瀏覽器下載 model.pt

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">9. 逐類召回率評估 (重點看 motorcycle)</b><br>用低 <code>conf</code> 在 test 集評估並印出每類 P/R。部署時調低偵測 <code>conf</code> 門檻可進一步提升召回。</div>

In [ ]:
m = YOLO(str(target))
metrics = m.val(data=str(FIXED_YAML), split='test', imgsz=1280, conf=0.001, device=0)

print(f"{'class':<12}{'P':>8}{'R':>8}{'mAP50':>8}")
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{m.names[c]:<12}{metrics.box.p[i]:>8.3f}{metrics.box.r[i]:>8.3f}{metrics.box.ap50[i]:>8.3f}")
print(f"\nall mAP50-95: {metrics.box.map:.3f}")